# NSE Bhavcopy Bulk Download

Downloads daily OHLCV data for all NIFTY500 PIT-union stocks (~800 tickers)
from NSE Bhavcopy archives (2012-01-01 to present).

Output: `nifty500_ohlcv.parquet` — per-ticker adjusted OHLCV ready for backtesting.

Data source: `nsearchives.nseindia.com` (official NSE equity bhavcopy)
Corporate actions: splits & bonuses applied backward from NSE API

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "yfinance", "requests", "beautifulsoup4", "lxml",
    "scikit-learn", "scipy", "ta", "arch", "pyarrow"])
print("Dependencies installed.")

In [ ]:
import shutil, os, sys

_CODE_CANDIDATES = [
    "/kaggle/input/centurion-core/centurion_core",
    "/kaggle/input/centurion-core",
    "/kaggle/input/datasets/srees16/centurion-core/centurion_core",
    "/kaggle/input/datasets/srees16/centurion-core",
]
src = None
for _candidate in _CODE_CANDIDATES:
    if os.path.isdir(_candidate) and os.path.isdir(os.path.join(_candidate, "services")):
        src = _candidate
        break
if src is None:
    for root, dirs, files in os.walk("/kaggle/input"):
        if "services" in dirs and ("cloud" in dirs or "runners" in dirs):
            src = root
            break
        if "centurion_core" in dirs:
            candidate = os.path.join(root, "centurion_core")
            if os.path.isdir(os.path.join(candidate, "services")):
                src = candidate
                break
if src is None:
    import subprocess
    subprocess.run(["find", "/kaggle/input", "-maxdepth", "6", "-type", "d"], timeout=10)
    raise FileNotFoundError("centurion_core/ not found under /kaggle/input")

print(f"Found source: {src}")
dst = "/kaggle/working/centurion_core"
shutil.copytree(src, dst, dirs_exist_ok=True)
print(f"Copied {src} -> {dst}")

sys.path.insert(0, "/kaggle/working")
os.chdir("/kaggle/working")
print("Ready.")

In [ ]:
import time
from datetime import date
from centurion_core.services.bhavcopy_fetcher import download_full_history

print("=" * 70)
print("  Phase 1: Bulk download NSE Bhavcopy files (2012-01-01 to today)")
print("=" * 70)

t0 = time.time()
n_downloaded = download_full_history(
    start=date(2012, 1, 1),
    end=date.today(),
    max_workers=4,
)
elapsed = time.time() - t0
print(f"\nPhase 1 complete: {n_downloaded} files in {elapsed/60:.1f} min")

In [ ]:
import time
import pandas as pd
from datetime import date
from centurion_core.kite_connect.nse.nse_universe import get_nse_universe_pit_union
from centurion_core.services.bhavcopy_fetcher import (
    fetch_ohlcv_batch, fetch_corporate_actions, adjust_ohlcv
)

print("=" * 70)
print("  Phase 2: Collate per-ticker adjusted OHLCV")
print("=" * 70)

# Get NIFTY500 PIT union tickers
pit_symbols = get_nse_universe_pit_union()
print(f"NIFTY500 PIT union: {len(pit_symbols)} symbols")

# Convert to .NS format for consistency with yfinance
tickers = [f"{s}.NS" for s in pit_symbols]

t0 = time.time()
START = date(2012, 1, 1)
END = date.today()

# Batch fetch raw OHLCV from cached bhavcopy files
print(f"Fetching raw OHLCV for {len(tickers)} tickers ({START} to {END})...")
raw_batch = fetch_ohlcv_batch(tickers, start=START, end=END)
print(f"Got data for {len(raw_batch)} / {len(tickers)} tickers")
elapsed = time.time() - t0
print(f"Raw OHLCV extraction: {elapsed/60:.1f} min")

# Apply corporate action adjustments per ticker
print(f"\nApplying corporate action adjustments...")
t1 = time.time()
adjusted = {}
ca_count = 0
ca_errors = 0
for i, (ticker, df) in enumerate(raw_batch.items()):
    try:
        adj_df = adjust_ohlcv(df, ticker, start=START, end=END)
        adjusted[ticker] = adj_df
        ca = fetch_corporate_actions(ticker, start=START, end=END)
        if not ca.empty:
            ca_count += 1
    except Exception as e:
        adjusted[ticker] = df  # use raw if adjustment fails
        ca_errors += 1
    if (i + 1) % 100 == 0:
        print(f"  [{i+1}/{len(raw_batch)}] adjusted={len(adjusted)} ca_found={ca_count} errors={ca_errors}")

elapsed2 = time.time() - t1
print(f"Corporate actions: {ca_count} tickers had splits/bonuses, {ca_errors} errors ({elapsed2/60:.1f} min)")

In [ ]:
import pandas as pd
import os

print("=" * 70)
print("  Phase 3: Save collated data")
print("=" * 70)

# Build a single long-format DataFrame: ticker × date × OHLCV
frames = []
for ticker, df in adjusted.items():
    if df.empty:
        continue
    chunk = df.copy()
    chunk["Ticker"] = ticker
    chunk = chunk.reset_index()  # Date becomes a column
    frames.append(chunk)

combined = pd.concat(frames, ignore_index=True)
combined.sort_values(["Ticker", "Date"], inplace=True)
combined.reset_index(drop=True, inplace=True)

print(f"Combined shape: {combined.shape}")
print(f"Tickers: {combined['Ticker'].nunique()}")
print(f"Date range: {combined['Date'].min()} to {combined['Date'].max()}")
print(f"Memory: {combined.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print()

# Save as parquet (efficient columnar format)
out_path = "/kaggle/working/nifty500_ohlcv.parquet"
combined.to_parquet(out_path, index=False, compression="snappy")
size_mb = os.path.getsize(out_path) / 1e6
print(f"Saved: {out_path} ({size_mb:.1f} MB)")

# Also save as CSV for easy inspection
csv_path = "/kaggle/working/nifty500_ohlcv.csv.gz"
combined.to_csv(csv_path, index=False, compression="gzip")
csv_size = os.path.getsize(csv_path) / 1e6
print(f"Saved: {csv_path} ({csv_size:.1f} MB)")

# Summary stats
print()
print("=" * 70)
print("  Summary")
print("=" * 70)
print(f"  Tickers:        {combined['Ticker'].nunique()}")
print(f"  Total rows:     {len(combined):,}")
print(f"  Date range:     {combined['Date'].min().date()} to {combined['Date'].max().date()}")
print(f"  Parquet size:   {size_mb:.1f} MB")
print(f"  CSV.gz size:    {csv_size:.1f} MB")

# Per-ticker row counts
counts = combined.groupby('Ticker').size()
print(f"  Min days/ticker: {counts.min()}")
print(f"  Max days/ticker: {counts.max()}")
print(f"  Median days:     {counts.median():.0f}")
print(f"  Tickers < 250d:  {(counts < 250).sum()}")